In [ ]:
%load_ext watermark


In [ ]:
import os

from matplotlib import pyplot as plt
import numpy as np
import pandas as pd


In [ ]:
%watermark -diwmuv -iv


In [ ]:
teeplot_subdir = os.environ.get("NOTEBOOK_NAME", "2026-03-23-d2h-memcpy-diagnostics")
teeplot_subdir


## Fetch Data


In [ ]:
url = "https://github.com/user-attachments/files/26186490/bookends.csv"
df = pd.read_csv(url)
print(f"{len(df)} mismatched bookend records")
df.head()


## Parse Bookends and Classify Parity Errors


In [ ]:
df["start_bookend"] = df["data_hex"].str[:8].apply(int, base=16)
df["end_bookend"] = df["data_hex"].str[-8:].apply(int, base=16)
df["diff"] = df["end_bookend"] - df["start_bookend"]

df["parity_error"] = np.where(
    df["diff"] == 512,
    "+1 parity",
    np.where(df["diff"] == -512, "-1 parity", "other"),
)
print(df["parity_error"].value_counts())
df.head()


## Spatial Coordinates


In [ ]:
nRow = 1170
nCol = 755
df["x"] = df["position"] // nCol  # row
df["y"] = df["position"] % nCol  # column

print(f"Row range: {df['x'].min()}--{df['x'].max()}")
print(f"Col range: {df['y'].min()}--{df['y'].max()}")


## Plot 1 --- Spatial Distribution


In [ ]:
fig, ax = plt.subplots(figsize=(6, 9))
ax.scatter(df["y"], df["x"], s=1, alpha=0.4, color="tab:red")
ax.set_xlim(0, nCol)
ax.set_ylim(nRow, 0)  # invert so row 0 is at top
ax.set_aspect("equal")
ax.set_xlabel("Column")
ax.set_ylabel("Row")
ax.set_title(f"Spatial Distribution of Bookend Mismatches (n={len(df):,})")
fig.tight_layout()
plt.show()


## Plot 2 --- Temporal Distribution Across Layers


In [ ]:
counts = df.groupby(["layer", "parity_error"]).size().unstack(fill_value=0)
# ensure both columns exist
for col in ["+1 parity", "-1 parity"]:
    if col not in counts.columns:
        counts[col] = 0
counts = counts[["+1 parity", "-1 parity"]]

total_per_layer = counts.sum(axis=1)
frac_plus1 = (counts["+1 parity"] / total_per_layer * 100).fillna(0)

fig, ax1 = plt.subplots(figsize=(12, 5))
counts.plot.bar(
    stacked=True,
    ax=ax1,
    color=["tab:blue", "tab:orange"],
    width=1.0,
    edgecolor="none",
)
ax1.set_xlabel("Layer")
ax1.set_ylabel("Mismatch Count")
ax1.set_title("Bookend Mismatches per Layer by Parity Error Type")
ax1.legend(loc="upper left")

# thin out x tick labels
tick_spacing = 50
tick_positions = range(0, len(counts), tick_spacing)
ax1.set_xticks(list(tick_positions))
ax1.set_xticklabels([counts.index[i] for i in tick_positions], rotation=45)

ax2 = ax1.twinx()
ax2.plot(
    range(len(frac_plus1)),
    frac_plus1.values,
    color="black",
    linewidth=1,
    alpha=0.7,
    label="+1 parity fraction (%)",
)
ax2.set_ylabel("+1 Parity Fraction (%)")
ax2.set_ylim(0, 100)
ax2.legend(loc="upper right")

fig.tight_layout()
plt.show()
